In [1]:
#import
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import requests
import os
from gwosc.datasets import find_datasets
from gwosc import datasets
import time

Matplotlib is building the font cache; this may take a moment.


In [6]:
#getting all GWTC events
url = "https://gwosc.org/api/v2/catalogs/GWTC/events"
params = {"include-default-parameters": "true", "pagesize": 20}

all_events = []

while True:
    #Send a GET request to GWOSC with page number as a query parameter
    response = requests.get(url, params)
    #for page 1, this is the same thing as: https://gwosc.org/api/v2/catalogs/GWTC/events?include-default-parameters=true&pagesize=20&page=1
    
    # Parse the JSON body into a Python dictionary
    data = response.json()

    #grab event data, which is under "results," and add to running list
    all_events.extend(data["results"])

    # If there's no next page, stop looping
    if data["next"] is None:
        break

    # Otherwise advance to the next page
    url = data["next"] #next page url is under "next"
    params = {} #clearing because next url has all the parameters
    time.sleep(0.5)  #wait half a second between requests to not overload the server

print(f"\nTotal events fetched: {len(all_events)}") #for debugging


Total events fetched: 391


In [ ]:
#filtering for BBH events
def is_bbh(event):
    params = {p["name"]: p["best"] for p in event.get("default_parameters", [])} #return empty list if "default_parameters" is somehow missing
    #get the list under dictionary key "default paraemeters" in event, and we are traversing each item p in the list, which are dictionaries, and make take p["name"] the keys and p["best"] as values for this new dictionary params
    mass_one = params.get("mass_1_source", 0)  # returns 0 if missing, won't crash
    mass_two = params.get("mass_2_source", 0)  # returns 0 if missing, won't crash
    return mass_one > 3 and mass_two > 3

bbh_events = [e for e in all_events if is_bbh(e)] #filter
print(f"Total events: {len(all_events)}") #for debugging
print(f"BBH events: {len(bbh_events)}") #for debugging
for e in bbh_events[:5]:
    print(e["name"]) #for debugging

KeyError: 'mass_1_source'